# 28. ViT 이미지 분류 실습

이 노트북은 `27_Vision_Transformer_ViT_핵심_아이디어.ipynb` 다음 단계로, ViT 기반 이미지 분류의 입력 처리와 출력 해석 흐름을 실습합니다.

실제 사전학습 ViT를 사용할 때도 핵심 흐름은 같습니다. 이미지를 정해진 크기로 맞추고, patch token으로 바꾼 뒤, Transformer encoder의 class token 표현을 classification head에 넣어 class score를 얻습니다.

이번 노트북의 목표는 다음과 같습니다.

- ViT inference의 전처리 흐름을 이해합니다.
- patch 단위로 이미지가 어떻게 분해되는지 확인합니다.
- class score, softmax probability, top-k prediction을 해석합니다.
- CNN 기반 분류기와 ViT 기반 분류기의 결과 해석 차이를 정리합니다.

## 28-1. 준비

외부 이미지나 사전학습 가중치 없이 실행되도록 간단한 예제 이미지를 직접 만듭니다. 실제 프로젝트에서는 이 부분을 사용자의 이미지 파일과 사전학습 ViT 모델로 바꾸면 됩니다.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=3, suppress=True)
np.random.seed(42)

## 28-2. 실습 이미지 만들기

ViT는 보통 입력 이미지를 정해진 크기로 resize하고 normalization한 뒤 사용합니다. 여기서는 작은 도형 이미지로 전처리 흐름을 확인합니다.

In [ ]:
image_path = None  # 예: 'data/my_image.jpg'

if image_path is not None and Path(image_path).exists():
    image = Image.open(image_path).convert('RGB')
else:
    image = Image.new('RGB', (224, 224), color=(232, 238, 245))
    draw = ImageDraw.Draw(image)
    draw.rectangle((35, 110, 190, 170), fill=(60, 110, 180))
    draw.ellipse((80, 50, 145, 115), fill=(230, 120, 60))
    draw.polygon([(60, 170), (112, 125), (165, 170)], fill=(70, 170, 110))

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title('입력 이미지')
plt.axis('off')
plt.show()

## 28-3. ViT 전처리 흐름

사전학습 ViT는 보통 다음 전처리를 기대합니다.

```text
PIL image -> resize/crop -> RGB tensor -> normalize -> patch embedding
```

여기서는 ImageNet 계열 모델에서 자주 쓰는 평균과 표준편차로 normalization하는 예시를 사용합니다.

In [ ]:
target_size = 224
resized = image.resize((target_size, target_size))
arr = np.asarray(resized).astype(np.float32) / 255.0

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
normalized = (arr - mean) / std

print('resized image shape:', arr.shape)
print('normalized min/max:', normalized.min().round(3), normalized.max().round(3))

## 28-4. Patch로 나누기

ViT-B/16처럼 patch size가 16인 모델은 `224 x 224` 이미지를 `14 x 14 = 196`개 patch로 나눕니다. 여기에 class token이 붙으면 Transformer encoder 입력 token 수는 197개가 됩니다.

In [ ]:
patch_size = 16
num_patches_per_side = target_size // patch_size
num_patches = num_patches_per_side ** 2

print('patch size:', patch_size)
print('patch grid:', num_patches_per_side, 'x', num_patches_per_side)
print('number of patch tokens:', num_patches)
print('tokens with CLS:', num_patches + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(resized)
ax.set_title('16 x 16 patch grid')
ax.set_xticks(range(0, target_size + 1, patch_size))
ax.set_yticks(range(0, target_size + 1, patch_size))
ax.grid(color='white', linewidth=0.8)
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.tick_params(length=0)
plt.show()

## 28-5. 예시 ViT 분류 출력 만들기

실제 사전학습 모델이 있다면 여기서 모델 forward 결과인 logits를 얻습니다. 이 노트북은 다운로드 없이 실행되도록 이미지의 색상 통계에서 예시 logits를 만들고, softmax와 top-k 해석만 동일하게 연습합니다.

In [ ]:
class_names = np.array(['airplane', 'car', 'flower', 'tree', 'dog'])

rgb_mean = arr.mean(axis=(0, 1))
brightness = arr.mean()
edge_hint = np.abs(np.diff(arr, axis=0)).mean() + np.abs(np.diff(arr, axis=1)).mean()

features = np.array([rgb_mean[0], rgb_mean[1], rgb_mean[2], brightness, edge_hint])
classifier = np.array([
    [1.2, 0.2, 0.1, 0.3, 0.1],
    [0.1, 0.7, 0.9, 1.1, 0.2],
    [0.2, 0.4, 1.3, 0.3, 0.5],
    [0.5, 0.6, 0.2, 0.8, 0.2],
    [0.2, 1.2, 0.1, 0.2, 0.7],
])
bias = np.array([0.1, 0.0, 0.05, -0.02, 0.03])
logits = features @ classifier + bias

def softmax(x):
    x = x - x.max()
    exp_x = np.exp(x)
    return exp_x / exp_x.sum()

prob = softmax(logits)
topk = np.argsort(prob)[::-1][:3]

print('logits:', logits.round(3))
for rank, idx in enumerate(topk, start=1):
    print(f'top {rank}: {class_names[idx]} ({prob[idx]:.3f})')

In [ ]:
plt.figure(figsize=(7, 3.5))
bars = plt.bar(class_names, prob, color=['#64748b', '#3b82f6', '#ef4444', '#22c55e', '#f59e0b'])
plt.ylim(0, 1)
plt.title('예시 ViT class probability')
plt.ylabel('probability')

for bar, p in zip(bars, prob):
    plt.text(bar.get_x() + bar.get_width() / 2, p + 0.02, f'{p:.2f}', ha='center')

plt.show()

## 28-6. 실제 사전학습 ViT를 쓸 때의 코드 형태

네트워크와 가중치 캐시가 준비된 환경에서는 torchvision 모델을 사용할 수 있습니다. 단, 아래 코드는 환경에 따라 가중치 다운로드가 필요할 수 있으므로 여기서는 자동 실행하지 않습니다.

```python
from torchvision.models import vit_b_16, ViT_B_16_Weights

weights = ViT_B_16_Weights.DEFAULT
model = vit_b_16(weights=weights).eval()
preprocess = weights.transforms()

x = preprocess(image).unsqueeze(0)
with torch.no_grad():
    logits = model(x)
prob = logits.softmax(dim=1)
```

중요한 것은 모델 종류가 달라도 `전처리 -> logits -> softmax -> top-k 해석` 흐름은 동일하다는 점입니다.

## 정리

- ViT inference에서는 입력 크기, normalization, patch size가 중요합니다.
- `224 x 224` 이미지와 `16 x 16` patch를 쓰면 patch token은 196개입니다.
- class token까지 포함하면 Transformer 입력 token은 197개가 됩니다.
- 분류 결과는 logits를 softmax로 바꾼 뒤 top-k class로 해석합니다.

다음 노트북 `29_DETR_객체_탐지_Transformer.ipynb`에서는 Transformer가 객체 탐지 문제를 어떻게 set prediction으로 바꾸는지 살펴봅니다.